In [ ]:
import os, json, numpy as np, torch
import torch.nn.functional as F

from evaluator_binary import evaluate_binary, compute_threshold
from evaluator_multiclass import evaluate_multiclass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
@torch.no_grad()
def collect_preds(model, loader, modality, device=DEVICE):
    """
    returns:
      y_true: (N,) int
      y_prob: (N,C) float
      sites : (N,) str
    """
    model.eval()
    y_true, y_prob, sites = [], [], []

    for batch in loader:
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device, non_blocking=True)

        if modality == "image":
            logits = model(task_type="image", x_img=batch["img"])
        elif modality == "tabular":
            logits = model(task_type="tabular", x_tab=batch["ehr"])
        else:
            raise ValueError("modality must be 'image' or 'tabular'")

        # softmax 概率
        if logits.shape[-1] == 1:
            prob = torch.sigmoid(logits).view(-1, 1)
        else:
            prob = F.softmax(logits, dim=-1)

        y_true.extend(batch["label"].long().cpu().numpy().tolist())
        y_prob.extend(prob.cpu().numpy().tolist())
        sites.extend(batch.get("client_id", ["global"] * len(batch["label"])))

    return np.array(y_true, int), np.array(y_prob, float), np.array(sites)


In [ ]:
def macro_micro(vals, weights=None):
    arr = np.array(vals, float)
    macro = float(arr.mean()) if len(arr) else float("nan")
    micro = float(np.average(arr, weights=weights)) if (weights is not None and len(arr)) else macro
    std   = float(arr.std()) if len(arr) else float("nan")
    rng   = float(arr.max() - arr.min()) if len(arr) else float("nan")
    return {"macro": macro, "micro": micro, "std": std, "range": rng}

def bootstrap_ci(stat_fn, y_true, y_prob, n_boot=1000, alpha=0.05, seed=2025):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        stats.append(stat_fn(y_true[idx], y_prob[idx]))
    low, high = np.quantile(stats, [alpha/2, 1-alpha/2])
    return float(low), float(high)


In [ ]:
def eval_binary_and_save(model, loader, modality, run_dir, split="test",
                         recall_target=0.80, do_ci=True):
    from sklearn.metrics import roc_auc_score, average_precision_score

    y_true, y_prob2, sites = collect_preds(model, loader, modality)
    y_prob = y_prob2[:, 1]  # 正类概率

    th_youden = compute_threshold(y_true, y_prob, method="youden")
    rep_youden, _ = evaluate_binary(y_true, y_prob, threshold=th_youden)

    th_rec   = compute_threshold(y_true, y_prob, method=f"fix_recall={recall_target}")
    rep_rec, _ = evaluate_binary(y_true, y_prob, threshold=th_rec)

    rep_050, _ = evaluate_binary(y_true, y_prob, threshold=0.5)

    # 分站点
    per_site, sizes = {}, {}
    for s in np.unique(sites):
        m = (sites == s)
        if m.sum() < 2: continue
        th_s = compute_threshold(y_true[m], y_prob[m], method="youden")
        rep_s, _ = evaluate_binary(y_true[m], y_prob[m], threshold=th_s)
        per_site[str(s)] = rep_s
        sizes[str(s)]    = int(m.sum())

    aurocs = [per_site[k]["AUROC"] for k in per_site]
    auprcs = [per_site[k]["AUPRC"] for k in per_site]
    w      = np.array([sizes[k] for k in per_site], float) if len(per_site) else None

    agg = {
        "AUROC": macro_micro(aurocs, w),
        "AUPRC": macro_micro(auprcs, w),
        "sizes": sizes
    }

    ci = None
    if do_ci:
        ci = {
            "AUROC": bootstrap_ci(lambda yt, yp: roc_auc_score(yt, yp), y_true, y_prob),
            "AUPRC": bootstrap_ci(lambda yt, yp: average_precision_score(yt, yp), y_true, y_prob)
        }

    os.makedirs(os.path.join(run_dir, "metrics"), exist_ok=True)
    out = {
        "SPLIT": split,
        "TASK": "binary",
        "GLOBAL": {
            "youden": {"overall": rep_youden, "threshold": float(th_youden)},
            f"recall@{recall_target:.2f}": {"overall": rep_rec, "threshold": float(th_rec)},
            "thr=0.5": {"overall": rep_050, "threshold": 0.5}
        },
        "PER_SITE": per_site,
        "AGGREGATION": agg,
        "CI": ci
    }
    save_path = os.path.join(run_dir, "metrics", f"{split}_binary_summary.json")
    with open(save_path, "w") as f: json.dump(out, f, indent=2)
    print("Saved →", save_path)
    return out


In [ ]:
def eval_multiclass_and_save(model, loader, modality, run_dir, split="test",
                             num_classes=5, do_ci=True):
    y_true, y_probC, sites = collect_preds(model, loader, modality)
    overall, per_class = evaluate_multiclass(y_true, y_probC, num_classes=num_classes, topk=5)

    per_site, sizes = {}, {}
    for s in np.unique(sites):
        m = (sites == s)
        if m.sum() < 2: continue
        ov_s, _ = evaluate_multiclass(y_true[m], y_probC[m], num_classes=num_classes, topk=5)
        per_site[str(s)] = ov_s
        sizes[str(s)]    = int(m.sum())

    auroc_macro_vals = [per_site[k]["AUROC_macro"] for k in per_site]
    auprc_macro_vals = [per_site[k]["AUPRC_macro"] for k in per_site]
    w = np.array([sizes[k] for k in per_site], float) if len(per_site) else None

    agg = {
        "AUROC_macro": macro_micro(auroc_macro_vals, w),
        "AUPRC_macro": macro_micro(auprc_macro_vals, w),
        "sizes": sizes
    }

    ci = None
    if do_ci:
        rng = np.random.default_rng(2025)
        n = len(y_true)
        pred = y_probC.argmax(axis=1)
        stats = []
        for _ in range(1000):
            idx = rng.integers(0, n, n)
            stats.append((pred[idx] == y_true[idx]).mean())
        low, high = np.quantile(stats, [0.025, 0.975])
        ci = {"Top1": (float(low), float(high))}

    os.makedirs(os.path.join(run_dir, "metrics"), exist_ok=True)
    out = {
        "SPLIT": split,
        "TASK": f"{num_classes}-class",
        "GLOBAL": {"overall": overall, "per_class": per_class},
        "PER_SITE": per_site,
        "AGGREGATION": agg,
        "CI": ci
    }
    save_path = os.path.join(run_dir, "metrics", f"{split}_{num_classes}class_summary.json")
    with open(save_path, "w") as f: json.dump(out, f, indent=2)
    print("Saved →", save_path)
    return out
